In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import kagglehub
from pathlib import Path

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection/"

FILE_LOCATION = Path(ROOT)

TRAIN_CSV = FILE_LOCATION/"train.csv"
TRAIN_SERIES_DIR = FILE_LOCATION/"train_series"
TEST_CSV = FILE_LOCATION/"test.csv"
TEST_SERIES_DIR = FILE_LOCATION/"train_series"


# The 12 target conditions
TARGET_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture"
]

# Quick overview — just list the top-level files, not every DICOM
print("Top-level contents:")
for item in sorted(FILE_LOCATION.iterdir()):
    if not item.is_file():
        n_children = len(list(item.iterdir()))
        print(f"  FOLDER: {item.name}/  ({n_children} items)")
    else:
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  FILE:   {item.name}  ({size_mb:.1f} MB)")

Top-level contents:
  FILE:   sample_submission.csv  (0.0 MB)
  FILE:   test.csv  (0.0 MB)
  FOLDER: test_series/  (3 items)
  FILE:   test_series.csv  (0.0 MB)
  FILE:   train.csv  (5.4 MB)
  FOLDER: train_series/  (4407 items)
  FILE:   train_series.csv  (3.3 MB)


In [2]:
train = pd.read_csv(f"{ROOT}/train.csv")

# --- sanity check: do the names actually match the file? ---
missing = [c for c in TARGET_COLS if c not in train.columns]
assert not missing, f"not in train.csv: {missing}"
ID_COLS = [c for c in train.columns if c not in TARGET_COLS]
print("shape:", train.shape)
print("ID columns:", ID_COLS)

# --- how much supervision ---
print("\n", train[TARGET_COLS].notna().all(axis=1).sum(), "fully labeled studies")
print(train[TARGET_COLS].isna().all(axis=1).sum(), "unlabeled studies")
print("\nnulls per target:\n", train[TARGET_COLS].isna().sum())

# --- binary or graded? check before treating means as prevalence ---
print("\ndistinct values per target:\n", train[TARGET_COLS].nunique().sort_values())
print("\ndtypes:\n", train[TARGET_COLS].dtypes)

# --- prevalence ---
print("\nprevalence:\n", train[TARGET_COLS].mean().sort_values().round(4))

# --- co-occurrence ---
print("\ncorrelation:\n", train[TARGET_COLS].corr().round(2).to_string())

# --- findings per study ---
print("\nfindings per study:\n", train[TARGET_COLS].sum(axis=1).value_counts().sort_index())

shape: (4407, 14)
ID columns: ['StudyInstanceUID', 'Report']

 58 fully labeled studies
4349 unlabeled studies

nulls per target:
 ACL                 4349
MCL                 4349
Medial Meniscus     4349
Lateral Meniscus    4349
Medial OA           4349
Lateral OA          4349
PF OA               4349
Effusion            4349
Synovitis           4349
Baker's             4349
Contusion           4349
Fracture            4349
dtype: int64

distinct values per target:
 ACL                 2
MCL                 2
Medial Meniscus     2
Lateral Meniscus    2
Medial OA           2
Lateral OA          2
PF OA               2
Effusion            2
Synovitis           2
Baker's             2
Contusion           2
Fracture            2
dtype: int64

dtypes:
 ACL                 float64
MCL                 float64
Medial Meniscus     float64
Lateral Meniscus    float64
Medial OA           float64
Lateral OA          float64
PF OA               float64
Effusion            float64
Synovitis      

In [3]:
gold = train.dropna(subset=TARGET_COLS).copy()
print(gold.shape)                                   # (58, 14)
print(gold[TARGET_COLS].sum(axis=1).value_counts().sort_index())

(58, 14)
1.0     8
2.0     7
3.0    11
4.0     7
5.0     9
6.0     7
7.0     5
8.0     1
9.0     3
Name: count, dtype: int64


In [4]:
r = train["Report"]
print(r.isna().sum(), "missing reports")
print(r.str.len().describe())

# Script mix — first pass at the language problem
import re
def script(s):
    if not isinstance(s, str): return "NA"
    if re.search(r'[\u4e00-\u9fff]', s): return "CJK"
    if re.search(r'[\u0600-\u06ff]', s): return "Arabic"
    if re.search(r'[\u0400-\u04ff]', s): return "Cyrillic"
    if re.search(r'[\u0900-\u097f]', s): return "Devanagari"
    return "Latin"
print(train["Report"].map(script).value_counts())

# The 58 gold reports are gold in a second sense: they let you see
# exactly how report wording maps to an expert's binary call.
for _, row in gold.head(5).iterrows():
    pos = [c for c in TARGET_COLS if row[c] == 1]
    print("\n" + "="*70)
    print("POSITIVE:", pos)
    print(row["Report"][:1200])

0 missing reports
count    4407.000000
mean     1097.905605
std       693.948925
min        52.000000
25%       587.500000
50%       977.000000
75%      1459.500000
max      4743.000000
Name: Report, dtype: float64
Report
Latin       4187
Cyrillic     220
Name: count, dtype: int64

POSITIVE: ['PF OA', 'Effusion']
Antecedentes Clínicos:
Esguince rodilla. [DATE].
Hallazgos:
No hay alteraciones de señal significativas de la médula ósea.
Ligamentos cruzados y colaterales dentro de límites normales.
Amputación marginal del cuerpo del menisco lateral. Menisco medial de morfología y señal
conservada, sin signos de rotura.
Cartílagos de los compartimentos femorotibiales sin alteraciones.
Fina úlcera condral focal de espesor total del aspecto inferior de la vertiente medial de la tróclea
femoral con mínimos cambios óseos secundarios. Fenómenos condrales reparativos de la región
central de la simple femoral. Cartílago rotuliano sin alteraciones.
Leve derrame articular. No hay quistes poplíteos p